
# BPMN 2.0 Process Dataset Builder for ML Training

Builds a clean, ML-ready dataset of BPMN 2.0 business processes from a raw
SAP-SAM-style CSV export, converting every valid process into the same JSON
schema used by `1972.json` (process / gateways / process_task / jobTasks /
job records).

**Pipeline stages**

| # | Stage | What it does |
|---|-------|---------------|
| 1 | Load & filter | Stream CSV(s), keep only valid BPMN 2.0 diagrams, keep only English processes |
| 2 | Sample | Reproducible random sample of `SAMPLE_SIZE` processes |
| 3 | Convert | Parse each diagram into tasks/gateways/flows, emit BPMN XML, synthesize missing fields |
| 4 | Validate | Schema + sanity checks on every converted record |
| 5 | Split & save | 80/20 train/eval split, one JSON file per process + `metadata.json` |
| 6 | Report | Summary statistics on kept/dropped/synthesized data |
| 7 | Self-test | Runs the whole pipeline end-to-end against a tiny synthetic CSV, so you can confirm everything works before pointing it at the real ~40GB dataset |

Every stage has its own function, its own error handling, and its own
progress bar, so the notebook can be re-run safely on partial data or
interrupted runs.



## 0. Setup

Dependencies: `tqdm` (progress bars), `pandas` (CSV streaming), `langdetect`
(language fallback check).


In [61]:

# If running in a fresh environment, uncomment:
# %pip install pandas tqdm langdetect --quiet

import copy
import json
import logging
import random
import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import pandas as pd
from tqdm.auto import tqdm

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("bpmn_pipeline")

print("Setup OK — imports loaded, langdetect available:", LANGDETECT_AVAILABLE)


Setup OK — imports loaded, langdetect available: True



## 1. Configuration

Everything the notebook needs lives in this one `Config` dataclass — no
magic paths buried in functions. Edit `input_path` / `output_path` for your
environment and re-run from here.

> **Note:** `input_path` must point at the folder that actually contains the
> `0.csv`, `10000.csv`, ... shards (e.g. `.../sap_sam_2022/models`), not the
> parent `sap_sam_2022` folder itself.


In [62]:

@dataclass
class Config:
    # --- Input ---
    # Accepts a single CSV path or a directory containing many CSV shards
    # (e.g. SAP-SAM's 0.csv, 10000.csv, 20000.csv, ...).
    input_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\data")
    csv_glob: str = "*.csv"
    csv_sep: str = ","           # confirmed: this export is comma-separated, quoted
    csv_encoding: str = "utf-8"
    chunksize: int = 2_000       # rows per chunk when streaming large files

    # --- Filtering ---
    target_language: str = "English"
    required_stencilset_substring: str = "bpmn2.0"
    # Only keep genuine flow-chart process diagrams, not choreography/
    # conversation/collaboration variants, per the SAP-SAM paper's
    # recommendation to treat these as distinct sub-populations.
    excluded_name_markers: tuple = ("choreography", "conversation")
    min_tasks: int = 2           # drop trivial/empty diagrams
    max_tasks: int = 200         # drop pathological outliers

    # --- Sampling ---
    sample_size: int = 3000
    random_seed: int = 42

    # --- Split ---
    train_ratio: float = 0.8

    # --- Output ---
    output_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed")
    train_dirname: str = "train"
    eval_dirname: str = "eval"
    metadata_filename: str = "metadata.json"
    stats_filename: str = "stats_report.json"

    # --- Synthetic data generation ---
    synth_seed: int = 7
    default_currency: str = "USD"
    hourly_rate_range: tuple = (15, 120)
    process_time_range_min: tuple = (5, 240)          # minutes
    rework_time_fraction_range: tuple = (0.05, 0.25)  # fraction of process time
    default_job_titles: tuple = (
        "Process Owner", "Department Manager", "Analyst",
        "Coordinator", "Specialist", "Clerk", "Supervisor",
    )
    default_org_name: str = "Synthetic Org"
    default_process_category_id: int = 1
    default_process_category_name: str = "Uncategorized"


def make_config(**overrides) -> "Config":
    '''Build a Config, apply overrides, and ensure its output dirs exist.'''
    cfg = Config(**overrides)
    (cfg.output_path / cfg.train_dirname).mkdir(parents=True, exist_ok=True)
    (cfg.output_path / cfg.eval_dirname).mkdir(parents=True, exist_ok=True)
    return cfg


CONFIG = make_config()
random.seed(CONFIG.random_seed)

print("Config OK")
print("  input_path :", CONFIG.input_path)
print("  output_path:", CONFIG.output_path)
print("  csv_sep    :", repr(CONFIG.csv_sep))
print("  sample_size:", CONFIG.sample_size)


Config OK
  input_path : C:\Users\yousu\Downloads\SAP\sap_sam_2022\data
  output_path: C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed
  csv_sep    : ','
  sample_size: 3000



## 2. Stage 1 — Load & Filter

The raw CSV has columns `Revision ID, Model ID, Organization ID, Datetime,
Model JSON, Description, Name, Type, Namespace` (plus, in some SAP-SAM
exports, a wide tail of empty padding columns which we simply ignore).

We stream the file in chunks (files can be 400MB+) rather than loading it
all into memory, and validate + filter each row independently so a single
malformed row never aborts the whole load.

Language filtering **cross-checks** the diagram's declared `properties.language`
against the actual name/description text via `langdetect`. The declared
field only reflects the workspace's UI locale at creation time — a user can
create an "English" workspace and label everything in Spanish — so trusting
it alone lets mislabeled rows through (confirmed empirically: doing so let
~640 wrong-language rows into a 5,001-row sample; cross-checking caught
~3,800 of them).


In [73]:

REQUIRED_COLUMNS = [
    "Revision ID", "Model ID", "Organization ID", "Datetime",
    "Model JSON", "Description", "Name", "Type", "Namespace",
]


def _is_valid_bpmn_json(raw_json: str, cfg: Config) -> Optional[dict]:
    '''Parse the Model JSON string and return the dict iff it's a genuine
    BPMN 2.0 process diagram matching the config's filters. Returns None
    (never raises) if the row should be dropped.'''
    if not raw_json or not isinstance(raw_json, str):
        return None
    try:
        model = json.loads(raw_json)
    except (json.JSONDecodeError, TypeError):
        return None

    stencilset = model.get("stencilset", {}) or {}
    namespace = (stencilset.get("namespace") or "") + (stencilset.get("url") or "")
    if cfg.required_stencilset_substring not in namespace.lower():
        return None

    # Must be an actual BPMN diagram stencil, not a process map/EPC/UML/DMN
    # that happens to reference the bpmn2.0 stencilset extensions.
    if model.get("stencil", {}).get("id") != "BPMNDiagram":
        return None

    return model


def _extract_language(model: dict, name: str, description: str, cfg: Config) -> Optional[str]:
    '''Cross-check the diagram's declared language property against the
    actual text content, since the declared property reflects the
    workspace's UI locale at creation time, not necessarily the language
    the modeler actually typed labels in.'''
    declared = (model.get("properties", {}) or {}).get("language")
    text = f"{name or ''} {description or ''}".strip()

    if not LANGDETECT_AVAILABLE or len(text) < 3:
        # No way to verify — fall back to trusting the declared field alone
        return declared

    try:
        detected_code = detect(text)
    except Exception:
        return declared

    detected_lang = "English" if detected_code == "en" else detected_code

    # Only trust "English" if detection agrees; for non-English declared
    # values we still return the detected language so filtering is accurate.
    if declared == cfg.target_language:
        return detected_lang if detected_code == "en" else detected_code
    return detected_lang


def _count_tasks(model: dict) -> int:
    '''Recursively count Task-stencil shapes in the diagram.'''
    count = 0

    def walk(shapes):
        nonlocal count
        for shape in shapes or []:
            stencil_id = (shape.get("stencil") or {}).get("id", "")
            if stencil_id == "Task":
                count += 1
            walk(shape.get("childShapes"))

    walk(model.get("childShapes"))
    return count


def load_and_filter(cfg: Config) -> tuple[list[dict], dict]:
    '''Stream every CSV under cfg.input_path, keep rows that pass all
    filters, and return (kept_rows, stats).'''
    input_path = Path(cfg.input_path)
    csv_files = sorted(input_path.glob(cfg.csv_glob)) if input_path.is_dir() else [input_path]

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found at {cfg.input_path} matching {cfg.csv_glob}")

    logger.info("Found %d CSV file(s) to scan.", len(csv_files))

    kept: list[dict] = []
    stats = {"scanned": 0, "bad_json": 0, "wrong_notation": 0,
              "wrong_language": 0, "excluded_variant": 0,
              "size_out_of_range": 0, "kept": 0}

    for csv_path in csv_files:
        try:
            reader = pd.read_csv(
                csv_path, sep=cfg.csv_sep, encoding=cfg.csv_encoding,
                usecols=lambda c: c in REQUIRED_COLUMNS,
                chunksize=cfg.chunksize, dtype=str, on_bad_lines="skip",
            )
        except Exception as exc:
            logger.warning("Skipping unreadable file %s: %s", csv_path.name, exc)
            continue

        for chunk in tqdm(reader, desc=f"Scanning {csv_path.name}", unit="chunk"):
            # pandas turns empty/missing cells into float NaN even with
            # dtype=str; NaN is truthy in Python, so `x or ""` doesn't
            # normalize it. Replace explicitly before any string ops.
            chunk = chunk.fillna("")

            for _, row in chunk.iterrows():
                stats["scanned"] += 1
                name = row.get("Name") or ""
                description = row.get("Description") or ""

                if any(m in name.lower() for m in cfg.excluded_name_markers):
                    stats["excluded_variant"] += 1
                    continue

                model = _is_valid_bpmn_json(row.get("Model JSON"), cfg)
                if model is None:
                    stats["bad_json"] += 1
                    continue

                lang = _extract_language(model, name, description, cfg)
                if lang != cfg.target_language:
                    stats["wrong_language"] += 1
                    continue

                n_tasks = _count_tasks(model)
                if not (cfg.min_tasks <= n_tasks <= cfg.max_tasks):
                    stats["size_out_of_range"] += 1
                    continue

                kept.append({
                    "revision_id": row.get("Revision ID"),
                    "model_id": row.get("Model ID"),
                    "organization_id": row.get("Organization ID"),
                    "datetime": row.get("Datetime"),
                    "name": name,
                    "description": description,
                    "model": model,
                    "n_tasks": n_tasks,
                })
                stats["kept"] += 1

    stats["wrong_notation"] = stats["bad_json"]
    logger.info("Load/filter complete: %s", stats)
    return kept, stats


print("Stage 1 OK — functions defined: _is_valid_bpmn_json, _extract_language, _count_tasks, load_and_filter")


Stage 1 OK — functions defined: _is_valid_bpmn_json, _extract_language, _count_tasks, load_and_filter



## 3. Stage 2 — Reproducible Sampling

Draw a fixed-seed random sample of `CONFIG.sample_size` from the filtered
pool. If fewer rows pass the filters than requested, sample everything
available and log a clear warning rather than failing silently.


In [74]:

def sample_processes(rows: list[dict], cfg: Config) -> list[dict]:
    rng = random.Random(cfg.random_seed)
    if len(rows) <= cfg.sample_size:
        logger.warning(
            "Only %d rows passed filtering (< requested sample_size=%d). "
            "Using all available rows.", len(rows), cfg.sample_size
        )
        sample = rows[:]
    else:
        sample = rng.sample(rows, cfg.sample_size)
    rng.shuffle(sample)
    logger.info("Sampled %d processes.", len(sample))
    return sample


print("Stage 2 OK — function defined: sample_processes")


Stage 2 OK — function defined: sample_processes



## 4. Stage 3 — Schema Conversion

Convert each sampled Signavio diagram into the `1972.json` schema:

```
process_id, company_id, process_code, process_name, process_overview,
process_category_id, process_status_id, process_version, bpmn_xml,
company{}, creator{}, processCategory{}, gateways[], process_task[]
```

The raw dataset only gives us diagram structure + name/description — it has
**no** company, job, cost, or timing data. Those fields are synthesized with
a seeded RNG (`cfg.synth_seed XOR process_id`) so the conversion is fully
reproducible.

`bpmn_xml` is a **simplified structural** BPMN 2.0 XML re-export (tasks,
events, gateways, sequence flows) — not a pixel-perfect reconstruction of
the original diagram, but valid, parseable BPMN 2.0 sufficient for
downstream flow analysis (cycle time, cost, critical path).


In [75]:

def _walk_all_shapes(shapes, parent_type=None):
    '''Yield (shape, stencil_id) for every shape in the tree, depth-first.'''
    for shape in shapes or []:
        stencil_id = (shape.get("stencil") or {}).get("id", "")
        yield shape, stencil_id
        yield from _walk_all_shapes(shape.get("childShapes"), stencil_id)


def _extract_flow_graph(model: dict) -> dict:
    '''Extract tasks, events, gateways and sequence flows from a Signavio
    BPMN JSON tree into flat lists, preserving resourceId linkage so we can
    rebuild ordering and gateway branches.'''
    tasks, gateways, events, flows = [], [], [], []

    for shape, stencil_id in _walk_all_shapes(model.get("childShapes")):
        rid = shape.get("resourceId")
        name = (shape.get("properties") or {}).get("name", "") or ""
        name = re.sub(r"\s+", " ", name).strip()
        outgoing = [o.get("resourceId") for o in shape.get("outgoing", [])]

        if stencil_id == "Task":
            tasks.append({"id": rid, "name": name or "Untitled Task", "outgoing": outgoing})
        elif "Gateway" in stencil_id:
            gateways.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif "Event" in stencil_id:
            events.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif stencil_id == "SequenceFlow":
            target = (shape.get("target") or {}).get("resourceId")
            flows.append({
                "id": rid,
                "name": (shape.get("properties") or {}).get("name", ""),
                "target": target,
            })

    return {"tasks": tasks, "gateways": gateways, "events": events, "flows": flows}


def _build_minimal_bpmn_xml(flow_graph: dict, process_name: str, process_code: str) -> str:
    '''Build a minimal, valid BPMN 2.0 XML document from the extracted
    flow graph. A compact structural approximation, not a full re-export.'''
    ET.register_namespace("bpmn", "http://www.omg.org/spec/BPMN/20100524/MODEL")
    NS = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    definitions = ET.Element(f"{{{NS}}}definitions", {
        "id": f"Definitions_{process_code}",
        "targetNamespace": "http://synthetic.local/bpmn",
    })
    process_el = ET.SubElement(definitions, f"{{{NS}}}process", {
        "id": f"Process_{process_code}", "name": process_name, "isExecutable": "false",
    })

    for t in flow_graph["tasks"]:
        ET.SubElement(process_el, f"{{{NS}}}task", {"id": t["id"], "name": t["name"]})
    for e in flow_graph["events"]:
        tag = ("startEvent" if "Start" in e["stencil"]
               else "endEvent" if "End" in e["stencil"]
               else "intermediateCatchEvent")
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": e["id"], "name": e.get("name", "")})
    for g in flow_graph["gateways"]:
        tag = ("exclusiveGateway" if "Exclusive" in g["stencil"]
               else "parallelGateway" if "Parallel" in g["stencil"]
               else "inclusiveGateway" if "Inclusive" in g["stencil"]
               else "eventBasedGateway")
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": g["id"], "name": g.get("name", "")})
    for f in flow_graph["flows"]:
        if f["target"]:
            ET.SubElement(process_el, f"{{{NS}}}sequenceFlow", {
                "id": f["id"], "targetRef": f["target"],
                **({"name": f["name"]} if f["name"] else {}),
            })

    xml_bytes = ET.tostring(definitions, encoding="utf-8", xml_declaration=True)
    return xml_bytes.decode("utf-8")


def _synth_job(rng: random.Random, cfg: Config, job_id: int) -> dict:
    title = rng.choice(cfg.default_job_titles)
    return {
        "job_id": job_id,
        "jobCode": f"SYN-J-{job_id}",
        "job_level_id": rng.randint(1, 6),
        "hourlyRate": rng.randint(*cfg.hourly_rate_range),
        "maxHoursPerDay": 8,
        "description": f"Synthetic role: {title}",
        "name": title,
        "capacity_buffer": str(rng.choice([5, 10, 15, 20])),
        "days_per_week": "5",
        "hours_per_day": "8",
        "currencyType": cfg.default_currency,
    }


def _synth_gateways(flow_graph: dict, rng: random.Random) -> list[dict]:
    result = []
    for i, g in enumerate(flow_graph["gateways"], start=1):
        gtype = ("EXCLUSIVE" if "Exclusive" in g["stencil"]
                 else "PARALLEL" if "Parallel" in g["stencil"]
                 else "INCLUSIVE" if "Inclusive" in g["stencil"] else "EVENT_BASED")
        n_branches = max(2, len(g["outgoing"]))
        raw_probs = [rng.random() + 0.1 for _ in range(n_branches)]
        total = sum(raw_probs)
        probs = [round(p / total, 2) for p in raw_probs]

        branches = [{
            "id": 10_000 + i * 10 + b,
            "gateway_pk_id": 1000 + i,
            "is_default": b == 0,
            "target_task_id": None,
            "condition": g["name"] or f"branch_{b+1}",
            "end_event_name": None,
            "end_task_id": None,
            "connect_to_end": rng.random() < 0.3,
            "target_gateway_id": None,
            "probability": probs[b],
        } for b in range(n_branches)]

        result.append({
            "gateway_pk_id": 1000 + i,
            "gateway_type": gtype,
            "after_task_id": None,
            "name": g["name"] or f"Gateway {i}",
            "converge_at_task_id": None,
            "converge_gateway_name": "",
            "converge_to_end": False,
            "converge_at_gateway_id": None,
            "after_gateway_id": None,
            "branches": branches,
        })
    return result


def convert_to_schema(row: dict, process_id: int, cfg: Config) -> dict:
    '''Convert one filtered raw row into the 1972.json-style schema.
    Raises ValueError on unrecoverable structural problems so the caller
    can log-and-skip without corrupting the output set.'''
    model = row["model"]
    flow_graph = _extract_flow_graph(model)

    if not flow_graph["tasks"]:
        raise ValueError("No tasks extracted from diagram")

    rng = random.Random(cfg.synth_seed ^ process_id)  # per-record but reproducible
    process_code = f"SYN-P-{process_id}"
    bpmn_xml = _build_minimal_bpmn_xml(flow_graph, row["name"] or process_code, process_code)

    process_tasks = []
    job_id_counter = 1
    for order, t in enumerate(flow_graph["tasks"], start=1):
        proc_time = rng.randint(*cfg.process_time_range_min)
        rework_frac = round(rng.uniform(*cfg.rework_time_fraction_range), 2)
        n_jobs = rng.choice([1, 1, 1, 2])  # mostly single-owner tasks
        job_tasks = []
        for _ in range(n_jobs):
            job = _synth_job(rng, cfg, job_id_counter)
            job_tasks.append({
                "job_id": job["job_id"],
                "task_id": 5000 + order,
                "role": rng.choice(["R", "A", "C", "I"]),
                "time_allocation_percentage": round(rng.uniform(1, 20), 2),
                "job": job,
            })
            job_id_counter += 1

        process_tasks.append({
            "process_task_id": 6000 + order,
            "process_id": process_id,
            "task_id": 5000 + order,
            "order": order,
            "child_process_id": None,
            "value_classification": rng.choice(["VA", "BVA", "NVA"]),
            "value_rationale": None,
            "bva_business_goal": None,
            "value_source": "synthetic",
            "task": {
                "task_id": 5000 + order,
                "task_code": f"SYN-T-{process_id}-{order}",
                "task_company_id": None,
                "task_name": t["name"],
                "task_overview": f"Synthetically enriched task extracted from diagram element {t['id']}.",
                "status_id": 1,
                "task_version": 0,
                "expected_process_time": proc_time,
                "expected_rework_time": round(proc_time * rework_frac),
                "expected_waiting_time": rng.choice([None, rng.randint(1, 30)]),
                "frequency_interval": 1,
                "frequency_period": rng.choice(["DAY", "WEEK", "MONTH"]),
                "occurrences": "1",
                "jobTasks": job_tasks,
            },
            "child_process": None,
        })

    total_time = sum(pt["task"]["expected_process_time"] for pt in process_tasks)

    return {
        "process_id": process_id,
        "company_id": 900_000 + process_id,
        "created_at": row["datetime"],
        "updated_at": row["datetime"],
        "capacity_requirement_minutes": total_time,
        "parent_process_id": None,
        "parent_task_id": None,
        "process_code": process_code,
        "process_name": row["name"] or process_code,
        "process_overview": row["description"] or "<p>No description provided in source data.</p>",
        "process_category_id": cfg.default_process_category_id,
        "process_status_id": 1,
        "process_version": 0,
        "bpmn_xml": bpmn_xml,
        "created_by": None,
        "updated_by": None,
        "PROCESS_STATUS": "CREATED",
        "bpmn_xml_updated_at": row["datetime"],
        "company": {
            "company_id": 900_000 + process_id,
            "companyCode": f"SYN-{process_id}",
            "name": cfg.default_org_name,
            "created_by": None,
            "org_type_id": 1,
        },
        "process": None,
        "creator": {"user_id": None, "name": "Synthetic Pipeline"},
        "processCategory": {
            "id": cfg.default_process_category_id,
            "description": "Auto-assigned category for synthetic dataset",
            "name": cfg.default_process_category_name,
        },
        "gateways": _synth_gateways(flow_graph, rng),
        "process_task": process_tasks,
        "_source": {
            "revision_id": row["revision_id"],
            "model_id": row["model_id"],
            "organization_id": row["organization_id"],
        },
    }


print("Stage 3 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, "
      "_synth_job, _synth_gateways, convert_to_schema")


Stage 3 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, _synth_job, _synth_gateways, convert_to_schema



## 5. Stage 4 — Validation

Every converted record is checked before it's allowed into the final
dataset. Failures are collected (not raised) so one bad record doesn't stop
the run; they're reported in the statistics summary at the end.


In [76]:

REQUIRED_TOP_LEVEL = [
    "process_id", "process_code", "process_name", "bpmn_xml",
    "gateways", "process_task",
]


def validate_record(record: dict) -> list[str]:
    '''Return a list of validation problems (empty list == valid).'''
    problems = []

    for key in REQUIRED_TOP_LEVEL:
        if key not in record or record[key] in (None, ""):
            problems.append(f"missing/empty field: {key}")

    if not record.get("process_task"):
        problems.append("process_task list is empty")
    else:
        for pt in record["process_task"]:
            task = pt.get("task", {})
            if task.get("expected_process_time", 0) <= 0:
                problems.append(f"task {task.get('task_code')} has non-positive process time")
            if not task.get("jobTasks"):
                problems.append(f"task {task.get('task_code')} has no job assignments")

    for gw in record.get("gateways", []):
        probs = [b["probability"] for b in gw.get("branches", [])]
        if probs and abs(sum(probs) - 1.0) > 0.05:
            problems.append(f"gateway {gw.get('name')} branch probabilities sum to {sum(probs):.2f}, not ~1.0")

    try:
        ET.fromstring(record["bpmn_xml"])
    except ET.ParseError as exc:
        problems.append(f"invalid bpmn_xml: {exc}")

    return problems


print("Stage 4 OK — function defined: validate_record")


Stage 4 OK — function defined: validate_record



## 6. Stage 5 — Convert + Validate the Full Sample

Runs conversion + validation over every sampled row, with a progress bar and
per-record error handling: a record that fails conversion or validation is
dropped and logged, never silently corrupted.


In [77]:

def convert_and_validate_all(sample: list[dict], cfg: Config) -> tuple[list[dict], dict]:
    converted = []
    conversion_errors = []
    validation_errors = []

    for i, row in enumerate(tqdm(sample, desc="Converting to schema", unit="proc")):
        process_id = 100_000 + i
        try:
            record = convert_to_schema(row, process_id, cfg)
        except Exception as exc:
            conversion_errors.append({"index": i, "name": row.get("name"), "error": str(exc)})
            continue

        problems = validate_record(record)
        if problems:
            validation_errors.append({"process_id": process_id, "problems": problems})
            continue

        converted.append(record)

    stats = {
        "attempted": len(sample),
        "converted_ok": len(converted),
        "conversion_failures": len(conversion_errors),
        "validation_failures": len(validation_errors),
        "conversion_error_samples": conversion_errors[:10],
        "validation_error_samples": validation_errors[:10],
    }
    logger.info(
        "Conversion complete: %d/%d succeeded (%d conversion errors, %d validation failures).",
        stats["converted_ok"], stats["attempted"],
        stats["conversion_failures"], stats["validation_failures"],
    )
    return converted, stats


print("Stage 5 OK — function defined: convert_and_validate_all")


Stage 5 OK — function defined: convert_and_validate_all



## 7. Stage 6 — Train/Eval Split & Save

An 80/20 split (configurable via `CONFIG.train_ratio`) with a fixed seed for
reproducibility. Each process is saved as its own `<process_code>.json` file
under `output_path/train/` or `output_path/eval/`, and a `metadata.json`
captures the run configuration, counts, and a manifest of which files ended
up in which split.


In [78]:

def split_and_save(records: list[dict], cfg: Config) -> dict:
    rng = random.Random(cfg.random_seed)
    shuffled = records[:]
    rng.shuffle(shuffled)

    split_idx = round(len(shuffled) * cfg.train_ratio)
    train_records, eval_records = shuffled[:split_idx], shuffled[split_idx:]

    manifest = {"train": [], "eval": []}

    for split_name, split_records in (("train", train_records), ("eval", eval_records)):
        out_dir = cfg.output_path / getattr(cfg, f"{split_name}_dirname")
        out_dir.mkdir(parents=True, exist_ok=True)
        for record in tqdm(split_records, desc=f"Saving {split_name}", unit="file"):
            filename = f"{record['process_code']}.json"
            out_path = out_dir / filename
            try:
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)
                manifest[split_name].append(filename)
            except OSError as exc:
                logger.error("Failed to write %s: %s", out_path, exc)

    metadata = {
        "config": {k: (str(v) if isinstance(v, Path) else v)
                   for k, v in vars(cfg).items()},
        "counts": {
            "train": len(manifest["train"]),
            "eval": len(manifest["eval"]),
            "total": len(manifest["train"]) + len(manifest["eval"]),
        },
        "manifest": manifest,
        "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }

    meta_path = cfg.output_path / cfg.metadata_filename
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    logger.info("Saved %d train / %d eval records. Metadata: %s",
                metadata["counts"]["train"], metadata["counts"]["eval"], meta_path)
    return metadata


print("Stage 6 OK — function defined: split_and_save")


Stage 6 OK — function defined: split_and_save



## 8. Stage 7 — Statistics Report

A single function that pulls together the counters from every stage into
one human-readable + machine-readable report, and writes it to
`output_path/stats_report.json`.


In [79]:

def build_stats_report(load_stats: dict, conv_stats: dict, split_meta: dict,
                        converted_records: list[dict], cfg: Config) -> dict:
    n_tasks_per_process = [len(r["process_task"]) for r in converted_records]
    n_gateways_per_process = [len(r["gateways"]) for r in converted_records]
    proc_times = [pt["task"]["expected_process_time"]
                  for r in converted_records for pt in r["process_task"]]

    def _avg(vals):
        return round(sum(vals) / len(vals), 2) if vals else 0

    report = {
        "stage_1_load_and_filter": load_stats,
        "stage_2_conversion": {
            k: v for k, v in conv_stats.items() if not k.endswith("_samples")
        },
        "stage_3_split": split_meta["counts"],
        "dataset_characteristics": {
            "avg_tasks_per_process": _avg(n_tasks_per_process),
            "min_tasks_per_process": min(n_tasks_per_process, default=0),
            "max_tasks_per_process": max(n_tasks_per_process, default=0),
            "avg_gateways_per_process": _avg(n_gateways_per_process),
            "avg_task_process_time_minutes": _avg(proc_times),
        },
        "sample_conversion_errors": conv_stats.get("conversion_error_samples", []),
        "sample_validation_errors": conv_stats.get("validation_error_samples", []),
    }

    stats_path = cfg.output_path / cfg.stats_filename
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"Rows scanned:              {load_stats['scanned']}")
    print(f"Passed BPMN/language/size filters: {load_stats['kept']}")
    print(f"Sampled for conversion:    {conv_stats['attempted']}")
    print(f"Successfully converted:    {conv_stats['converted_ok']}")
    print(f"  Conversion failures:     {conv_stats['conversion_failures']}")
    print(f"  Validation failures:     {conv_stats['validation_failures']}")
    print(f"Final train / eval split:  {split_meta['counts']['train']} / {split_meta['counts']['eval']}")
    print(f"Avg tasks per process:     {report['dataset_characteristics']['avg_tasks_per_process']}")
    print(f"Avg gateways per process:  {report['dataset_characteristics']['avg_gateways_per_process']}")
    print(f"Avg task time (min):       {report['dataset_characteristics']['avg_task_process_time_minutes']}")
    print(f"\nFull report written to: {stats_path}")
    print("=" * 60)

    return report


print("Stage 7 OK — function defined: build_stats_report")


Stage 7 OK — function defined: build_stats_report



## 9. Orchestration — `run_pipeline()`

Ties every stage together with top-level error handling: if a stage raises
an unrecoverable exception, it's logged clearly and the run stops rather
than continuing on corrupted state.


In [80]:

def run_pipeline(cfg: Config) -> dict:
    try:
        rows, load_stats = load_and_filter(cfg)
    except FileNotFoundError as exc:
        logger.error("Pipeline aborted at load stage: %s", exc)
        raise

    if not rows:
        logger.error("No rows passed filtering — check csv_sep / paths / filters in Config.")
        return {}

    sample = sample_processes(rows, cfg)
    converted, conv_stats = convert_and_validate_all(sample, cfg)

    if not converted:
        logger.error("No records survived conversion/validation — aborting before save.")
        return {}

    split_meta = split_and_save(converted, cfg)
    report = build_stats_report(load_stats, conv_stats, split_meta, converted, cfg)
    return report


print("Stage 8 OK — function defined: run_pipeline")


Stage 8 OK — function defined: run_pipeline



## 10. Self-Test — Verify the Pipeline Works End-to-End

This runs the **entire pipeline** against a small, self-contained synthetic
CSV (generated right here, no dependency on your real dataset), so you can
confirm every stage works correctly before pointing it at the real ~40GB
corpus. It exercises every filter (valid BPMN, wrong notation, wrong
language, choreography exclusion, malformed JSON) and asserts the expected
outcome at each stage.

If this cell passes, the pipeline logic is verified — any issues in the
real run come from data characteristics, not pipeline bugs.


In [81]:

import csv as _csv
import shutil
import tempfile


def _make_test_bpmn_model(lang="English", n_tasks=3):
    '''Build a minimal but structurally valid Signavio BPMN JSON model for testing.'''
    shapes = [{
        "resourceId": "start1", "properties": {"name": "Start"},
        "stencil": {"id": "StartNoneEvent"}, "outgoing": [{"resourceId": "task1"}],
        "childShapes": [],
    }]
    for i in range(1, n_tasks + 1):
        nxt = f"task{i+1}" if i < n_tasks else "gw1"
        shapes.append({
            "resourceId": f"task{i}", "properties": {"name": f"Task {i}"},
            "stencil": {"id": "Task"}, "outgoing": [{"resourceId": nxt}], "childShapes": [],
        })
    shapes += [
        {"resourceId": "gw1", "properties": {"name": "Decision?"},
         "stencil": {"id": "Exclusive_Databased_Gateway"},
         "outgoing": [{"resourceId": "sf1"}, {"resourceId": "sf2"}], "childShapes": []},
        {"resourceId": "sf1", "properties": {"name": "Yes"}, "stencil": {"id": "SequenceFlow"},
         "outgoing": [], "target": {"resourceId": "end1"}, "childShapes": []},
        {"resourceId": "sf2", "properties": {"name": "No"}, "stencil": {"id": "SequenceFlow"},
         "outgoing": [], "target": {"resourceId": "end1"}, "childShapes": []},
        {"resourceId": "end1", "properties": {"name": "End"}, "stencil": {"id": "EndNoneEvent"},
         "outgoing": [], "childShapes": []},
    ]
    return {
        "resourceId": "canvas", "properties": {"language": lang},
        "stencil": {"id": "BPMNDiagram"},
        "stencilset": {"namespace": "http://b3mn.org/stencilset/bpmn2.0#",
                       "url": "/stencilsets/bpmn2.0/bpmn2.0.json"},
        "childShapes": shapes,
    }


def _build_test_csv(path: Path) -> int:
    '''Write a small synthetic CSV exercising every filter path.
    Returns the number of rows that SHOULD be kept after filtering.'''
    rows, expected_kept = [], 0

    for i in range(5):  # valid English BPMN -> kept
        rows.append({
            "Revision ID": f"rev{i}", "Model ID": f"mod{i}", "Organization ID": f"org{i}",
            "Datetime": "2020-01-01 10:00:00",
            "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=3 + i % 2)),
            "Description": "A test process", "Name": f"Test Process {i}",
            "Type": "", "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
        })
        expected_kept += 1

    rows.append({  # declared English but actually Spanish text -> should be dropped
        "Revision ID": "reves", "Model ID": "modes", "Organization ID": "orges",
        "Datetime": "2020-01-01 10:00:00",
        "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=3)),
        "Description": "Elaborar productos para el cliente final",
        "Name": "Elaborar productos", "Type": "", "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
    })

    rows.append({  # non-BPMN notation -> dropped
        "Revision ID": "revmap", "Model ID": "modmap", "Organization ID": "orgmap",
        "Datetime": "2020-01-01 10:00:00",
        "Model JSON": json.dumps({"resourceId": "canvas", "stencil": {"id": "Diagram"},
                                   "stencilset": {"namespace": "http://www.signavio.com/stencilsets/processmap#"},
                                   "childShapes": []}),
        "Description": "", "Name": "Process Map", "Type": "",
        "Namespace": "http://www.signavio.com/stencilsets/processmap#",
    })

    rows.append({  # malformed JSON -> dropped
        "Revision ID": "revbad", "Model ID": "modbad", "Organization ID": "orgbad",
        "Datetime": "2020-01-01 10:00:00", "Model JSON": "{not valid json",
        "Description": "", "Name": "Bad Process", "Type": "", "Namespace": "",
    })

    rows.append({  # choreography variant -> dropped by name marker
        "Revision ID": "revchor", "Model ID": "modchor", "Organization ID": "orgchor",
        "Datetime": "2020-01-01 10:00:00",
        "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=2)),
        "Description": "", "Name": "My Choreography Process", "Type": "",
        "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
    })

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=[
            "Revision ID", "Model ID", "Organization ID", "Datetime",
            "Model JSON", "Description", "Name", "Type", "Namespace",
        ])
        writer.writeheader()
        writer.writerows(rows)

    return expected_kept


def run_self_test() -> bool:
    '''Run the full pipeline against a tiny synthetic dataset and assert
    every stage behaves as expected. Returns True iff all checks pass.'''
    tmp_dir = Path(tempfile.mkdtemp(prefix="bpmn_pipeline_selftest_"))
    checks = []

    try:
        csv_path = tmp_dir / "0.csv"
        expected_kept = _build_test_csv(csv_path)

        test_cfg = make_config(
            input_path=tmp_dir,
            output_path=tmp_dir / "out",
            csv_sep=",",
            sample_size=100,  # more than available -> exercises the "use all rows" path
        )

        report = run_pipeline(test_cfg)

        checks.append(("pipeline returned a non-empty report", bool(report)))
        checks.append(("load_and_filter kept exactly the expected rows",
                        report["stage_1_load_and_filter"]["kept"] == expected_kept))
        checks.append(("wrong_language filter caught the mislabeled Spanish row",
                        report["stage_1_load_and_filter"]["wrong_language"] >= 1))
        checks.append(("excluded_variant filter caught the choreography row",
                        report["stage_1_load_and_filter"]["excluded_variant"] == 1))
        checks.append(("bad_json filter caught the malformed + non-BPMN rows",
                        report["stage_1_load_and_filter"]["bad_json"] == 2))
        checks.append(("all sampled rows converted with zero failures",
                        report["stage_2_conversion"]["conversion_failures"] == 0
                        and report["stage_2_conversion"]["validation_failures"] == 0))
        checks.append(("train + eval counts match converted total",
                        report["stage_3_split"]["total"] == report["stage_2_conversion"]["converted_ok"]))

        train_dir = test_cfg.output_path / test_cfg.train_dirname
        eval_dir = test_cfg.output_path / test_cfg.eval_dirname
        saved_files = list(train_dir.glob("*.json")) + list(eval_dir.glob("*.json"))
        checks.append(("output JSON files exist on disk", len(saved_files) == expected_kept))

        if saved_files:
            with open(saved_files[0], encoding="utf-8") as f:
                sample_record = json.load(f)
            checks.append(("saved record parses back as valid JSON with required fields",
                            all(k in sample_record for k in REQUIRED_TOP_LEVEL)))
            checks.append(("saved record's bpmn_xml is well-formed XML",
                            _xml_is_valid(sample_record["bpmn_xml"])))

        metadata_path = test_cfg.output_path / test_cfg.metadata_filename
        stats_path = test_cfg.output_path / test_cfg.stats_filename
        checks.append(("metadata.json was written", metadata_path.exists()))
        checks.append(("stats_report.json was written", stats_path.exists()))

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print("=" * 60)
    print("SELF-TEST RESULTS")
    print("=" * 60)
    all_passed = True
    for description, passed in checks:
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_passed = False
        print(f"  [{status}] {description}")
    print("=" * 60)
    print("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED — see above")
    print("=" * 60)
    return all_passed


def _xml_is_valid(xml_string: str) -> bool:
    try:
        ET.fromstring(xml_string)
        return True
    except ET.ParseError:
        return False


self_test_passed = run_self_test()


02:58:25 | INFO     | Found 1 CSV file(s) to scan.


Scanning 0.csv: 0chunk [00:00, ?chunk/s]

02:58:25 | INFO     | Load/filter complete: {'scanned': 9, 'bad_json': 2, 'wrong_notation': 2, 'wrong_language': 1, 'excluded_variant': 1, 'size_out_of_range': 0, 'kept': 5}
02:58:25 | WARNING  | Only 5 rows passed filtering (< requested sample_size=100). Using all available rows.
02:58:25 | INFO     | Sampled 5 processes.


Converting to schema:   0%|          | 0/5 [00:00<?, ?proc/s]

02:58:25 | INFO     | Conversion complete: 5/5 succeeded (0 conversion errors, 0 validation failures).


Saving train:   0%|          | 0/4 [00:00<?, ?file/s]

Saving eval:   0%|          | 0/1 [00:00<?, ?file/s]

02:58:25 | INFO     | Saved 4 train / 1 eval records. Metadata: C:\Users\yousu\AppData\Local\Temp\bpmn_pipeline_selftest_jbeixhfo\out\metadata.json


PIPELINE SUMMARY
Rows scanned:              9
Passed BPMN/language/size filters: 5
Sampled for conversion:    5
Successfully converted:    5
  Conversion failures:     0
  Validation failures:     0
Final train / eval split:  4 / 1
Avg tasks per process:     3.4
Avg gateways per process:  1.0
Avg task time (min):       137.59

Full report written to: C:\Users\yousu\AppData\Local\Temp\bpmn_pipeline_selftest_jbeixhfo\out\stats_report.json
SELF-TEST RESULTS
  [PASS] pipeline returned a non-empty report
  [PASS] load_and_filter kept exactly the expected rows
  [PASS] wrong_language filter caught the mislabeled Spanish row
  [PASS] excluded_variant filter caught the choreography row
  [PASS] bad_json filter caught the malformed + non-BPMN rows
  [PASS] all sampled rows converted with zero failures
  [PASS] train + eval counts match converted total
  [PASS] output JSON files exist on disk
  [PASS] saved record parses back as valid JSON with required fields
  [PASS] saved record's bpmn_xml is


## 11. Run on Real Data

Once the self-test above passes, run the real pipeline against your actual
dataset. This scans every CSV under `CONFIG.input_path` — for the full
SAP-SAM export (~103 files, ~40GB), expect this to take a while (order of
tens of minutes to an hour depending on disk speed), since every row's JSON
is parsed and every diagram is walked to count tasks.


In [82]:

assert self_test_passed, "Self-test failed — fix the pipeline before running on real data."

report = run_pipeline(CONFIG)


02:58:26 | INFO     | Found 103 CSV file(s) to scan.


Scanning 0.csv: 0chunk [00:00, ?chunk/s]

Scanning 10000.csv: 0chunk [00:00, ?chunk/s]

Scanning 100000.csv: 0chunk [00:00, ?chunk/s]

Scanning 1000000.csv: 0chunk [00:00, ?chunk/s]

...

Scanning 830000.csv: 0chunk [00:00, ?chunk/s]

Scanning 840000.csv: 0chunk [00:00, ?chunk/s]

Scanning 850000.csv: 0chunk [00:00, ?chunk/s]

Scanning 860000.csv: 0chunk [00:00, ?chunk/s]

ParserError: Error tokenizing data. C error: out of memory


## 12. Notes

- **Point `CONFIG.input_path`** at your SAP-SAM CSV directory (the folder
  containing `0.csv`, `10000.csv`, ...) before running Section 11.
- **Language filtering** cross-checks the diagram's own `properties.language`
  field against `langdetect` on the name/description text — trusting the
  declared field alone lets mislabeled rows through.
- **Synthetic fields** (company, job, cost, timing, RACI, gateway
  probabilities) are generated with a per-record seed
  (`cfg.synth_seed XOR process_id`), so re-running reproduces byte-identical
  synthetic data.
- **`bpmn_xml`** is a simplified structural re-export — valid, parseable
  BPMN 2.0 XML sufficient for flow analysis, not a pixel-perfect
  reconstruction of the original Signavio diagram.
- All error handling favors **skip-and-log** over **crash-the-run**.
- The **self-test in Section 10** is the fast way to verify any future code
  changes didn't break the pipeline — it runs in well under a second and
  needs no real data.
- To adjust the split ratio, sample size, or filters, edit the `Config`
  dataclass in Section 1 only.
